In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

# 1. Churn prediction project

## The problem of churn

*Churn* is when a customer stops using a service.
- A telecom company has many customers, and every month some of them decide to terminate their contracts and switch to another provider.
- Losing customers is expensive, so the company wants to identify the people who are likely to leave before they actually do.

The idea is simple: 
1. For every customer, we compute a score between 0 and 1 - the probability that this customer is about to churn. 
2. Then the company can target the customers with high scores and try to retain them, for example by sending an email with discounts or special promotions.

## Classification

The task is *binary classification*: we look at one customer and predict one of two possible outcomes.

For the $i$-th customer, we want our model $g$ to approximate the target $y$:
\begin{equation} g(x_i)\approx y_i \end{equation}

Here $x$ is the feature vector - everything we know about the customer - and $y$ is the target variable.

The target is binary, so $y$ belongs to {0,1}:
- 1 is the positive class: the customer churned
- 0 is the negative class: the customer stayed

The output of the model - the score - is the likelihood of churning.

We learn $g$ from historical data. We take the customers of the previous month - we already know what happened to them. For each of them we know the demographics, what services they use, what kind of contract they have and how much they pay, and we know whether they stayed or left. The model finds the patterns in this data, and then we apply it to the current customers to score them.

## The dataset

* Dataset: https://www.kaggle.com/blastchar/telco-customer-churn
* https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv

The whole module walks through this project step by step: we prepare the data, set up a validation framework, do EDA and feature importance analysis, then train a logistic regression model and use it to score customers.

# 2.Data preparation

Download the data, read it with pandas:
* Look at the data
* Make column names and values look uniform
* Check if all the columns read correctly
* Check if the churn variable needs any preparation

## Downloading and reading the data

In [2]:
data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'

* The '!' prefix runs a shell command 
* The '$' lets us use a Python variable inside a shell command
* The '-O' flag says how to name the downloaded file

In [3]:
!wget $data -O data-week-3.csv 

--2026-10-05 13:07:03--  https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 977501 (955K) [text/plain]
Saving to: ‘data-week-3.csv’

data-week-3.csv     100%[===================>] 954.59K  --.-KB/s    in 0.005s  

2026-10-05 13:07:03 (188 MB/s) - ‘data-week-3.csv’ saved [977501/977501]



In [4]:
df = pd.read_csv('data-week-3.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
df.head().T         #trasposta del dataframe per vedere tutte le colonne

,0,1,2,3,4
customerID,7590-VHVEG,5575-GNVDE,3668-QPYBK,7795-CFOCW,9237-HQITU
gender,Female,Male,Male,Male,Female
SeniorCitizen,0,0,0,0,0
Partner,Yes,No,No,No,No
Dependents,No,No,No,No,No
tenure,1,34,2,45,2
PhoneService,No,Yes,Yes,No,Yes
MultipleLines,No phone service,No,No,No phone service,No
InternetService,DSL,DSL,DSL,DSL,Fiber optic
OnlineSecurity,No,Yes,Yes,Yes,No


## Making everything uniform

The columns names are inconsistent, so the values (uppercase letters, spaces):

In [8]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

categorical_columns = list(df.dtypes[df.dtypes == 'str'].index)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

In [9]:
df.head().T

,0,1,2,3,4
customerid,7590-vhveg,5575-gnvde,3668-qpybk,7795-cfocw,9237-hqitu
gender,female,male,male,male,female
seniorcitizen,0,0,0,0,0
partner,yes,no,no,no,no
dependents,no,no,no,no,no
tenure,1,34,2,45,2
phoneservice,no,yes,yes,no,yes
multiplelines,no_phone_service,no,no,no_phone_service,no
internetservice,dsl,dsl,dsl,dsl,fiber_optic
onlinesecurity,no,yes,yes,yes,no


In [10]:
df.dtypes["contract"]

<StringDtype(storage='python', na_value=nan)>

## Fixing the totalcharges column

We notice that 'totalcharges' column is a string column, but it contains numbers.

Let's try to convert it to numbers:

In [11]:
pd.to_numeric(df.totalcharges)

ValueError: Unable to parse string "_" at position 488

This fails: in the original data, missing values in this column were encoded with a space.

Our normalization step replaced spaces with underscores, so the missing values became "_", and pandas cannot parse that as a number.

We can force the conversion in a way that unparseable strings are replaced with NaN:

In [12]:
tc = pd.to_numeric(df.totalcharges, errors = 'coerce')

Let's see how many missing values this creates:

In [13]:
tc.isnull().sum()

np.int64(11)

In [14]:
df[tc.isnull()][['customerid', 'totalcharges']]

,customerid,totalcharges
488,4472-lvygi,_
753,3115-czmzd,_
936,5709-lvoeq,_
1082,4367-nuyao,_
1340,1371-dwpaz,_
3331,7644-omvmy,_
3826,3213-vvolg,_
4380,2520-sgtta,_
5218,2923-arzlg,_
6670,4075-wkniu,_


These are customers who just joined - their tenure is small and they haven't been billed yet, so their total charges are not available. 

For them we simply fill the missing values with zero:

In [15]:
df.totalcharges = df.totalcharges.fillna(0)

Zero is not always the best replacement, but in this case it is acceptable: it matches the fact that these customers haven't paid anything yet.

## Encoding the target variable

The target 'churn' contains strings:

In [16]:
df.churn.head()

0     no
1     no
2    yes
3     no
4    yes
Name: churn, dtype: str

For classification we need numbers, so we compare the column with 'yes', producing a boolean series - and cast it to integers:

In [17]:
df.churn = (df.churn == 'yes').astype(int)

Now 'churn' is 1 for customers who churned and 0 for those who stayed. 

The data is ready for the next step: splitting it into train, validation and test sets.

# 3.Setting up the validation framework

Before training anything, we split the data into train, validation and test sets using Scikit-Learn, and we put the target variable aside so it cannot leak into the features.

## Splitting the data

In [18]:
from sklearn.model_selection import train_test_split

But we want the classic 60/20/20 split; the function only splits in two, so we do it in two steps.

- First we set aside the test set - 20% of all the data;
- The remaining 80% we call the full train set. Then we split it again into train and validation:

In [19]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

In [20]:
len(df_train), len(df_val), len(df_test)

(4225, 1409, 1409)

## Resetting the index

'train_test_split' shuffles the rows, so after splitting the row indexes are a random mixture of numbers.

We reset them:

In [21]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

## Isolating the target

The target for this project is the 'churn' column. We extract it from each dataframe as a NumPy array:

In [22]:
y_train = df_train.churn.values
y_val = df_val.churn.values
y_test = df_test.churn.values

del df_train['churn']
del df_val['churn']
del df_test['churn']

# 4. EDA

Now we explore the full training data (train and validation together):
* Check missing values
* Look at the target variable (churn)
* Look at numerical and categorical variables

In [23]:
df_full_train = df_full_train.reset_index(drop=True)

## Missing values

In [24]:
df_full_train.isnull().sum()

customerid          0
gender              0
seniorcitizen       0
partner             0
dependents          0
tenure              0
phoneservice        0
multiplelines       0
internetservice     0
onlinesecurity      0
onlinebackup        0
deviceprotection    0
techsupport         0
streamingtv         0
streamingmovies     0
contract            0
paperlessbilling    0
paymentmethod       0
monthlycharges      0
totalcharges        0
churn               0
dtype: int64

Everything is 0 - there are no missing values anywhere. We already took care of the only column that had them, totalcharges, during data preparation.

## Looking at the target: the churn rate

In [25]:
df_full_train.churn.value_counts(normalize=True)

churn
0    0.730032
1    0.269968
Name: proportion, dtype: float64

About 73% of the customers stayed and 27% churned. The percentage of ones - 0.269968, roughly 27% - is called the churn rate.

In [26]:
df_full_train.churn.mean()

np.float64(0.26996805111821087)

Same number. This works for any binary variable: its mean is always the fraction of ones.

## Numerical and categorical variables

Next we decide which columns are numerical and which are categorical. 

Most columns in this dataset contain strings, so they are categorical. Among the numeric ones, 'seniorcitizen' is really a categorical variable - it is just encoded as 0/1 - and 'churn is the target. 

That leaves three numerical variables:

In [27]:
numerical = ['tenure', 'monthlycharges', 'totalcharges']

And 16 categorical ones - everything except 'customerid' (an identifier, useless as a feature), the three numerical variables and the target:

In [28]:
categorical = [
    'gender',
    'seniorcitizen',
    'partner',
    'dependents',
    'phoneservice',
    'multiplelines',
    'internetservice',
    'onlinesecurity',
    'onlinebackup',
    'deviceprotection',
    'techsupport',
    'streamingtv',
    'streamingmovies',
    'contract',
    'paperlessbilling',
    'paymentmethod',
]

It is a good idea to check how many unique values each categorical variable has:

In [29]:
df_full_train[categorical].nunique()

gender              2
seniorcitizen       2
partner             2
dependents          2
phoneservice        2
multiplelines       3
internetservice     3
onlinesecurity      3
onlinebackup        3
deviceprotection    3
techsupport         3
streamingtv         3
streamingmovies     3
contract            3
paperlessbilling    2
paymentmethod       4
dtype: int64

Most variables have two or three values; 'paymentmethod' has four - it is the most diverse one. 

The number of categories matters later: it determines how many columns we get after one-hot encoding.

# 5. Feature importance: Churn rate and risk ratio

Feature importance analysis (part of EDA) - identifying which features affect our target variable

* Churn rate
* Risk ratio
* Mutual information - later

#### Churn rate

In [ ]:
df_full_train.head()

In [ ]:
churn_female = df_full_train[df_full_train.gender == 'female'].churn.mean()
churn_female

In [ ]:
churn_male = df_full_train[df_full_train.gender == 'male'].churn.mean()
churn_male

In [ ]:
global_churn = df_full_train.churn.mean()
global_churn

In [ ]:
global_churn - churn_female

In [ ]:
global_churn - churn_male

In [ ]:
df_full_train.partner.value_counts()

In [ ]:
churn_partner = df_full_train[df_full_train.partner == 'yes'].churn.mean()
churn_partner

In [ ]:
global_churn - churn_partner

In [ ]:
churn_no_partner = df_full_train[df_full_train.partner == 'no'].churn.mean()
churn_no_partner

In [ ]:
global_churn - churn_no_partner

#### Risk ratio

In [ ]:
churn_no_partner / global_churn

In [ ]:
churn_partner / global_churn

```
SELECT
    gender,
    AVG(churn),
    AVG(churn) - global_churn AS diff,
    AVG(churn) / global_churn AS risk
FROM
    data
GROUP BY
    gender;
```

In [ ]:
from IPython.display import display

In [ ]:
for c in categorical:
    print(c)
    df_group = df_full_train.groupby(c).churn.agg(['mean', 'count'])
    df_group['diff'] = df_group['mean'] - global_churn
    df_group['risk'] = df_group['mean'] / global_churn
    display(df_group)
    print()
    print()

## 3.6 Feature importance: Mutual information

Mutual information - concept from information theory, it tells us how much 
we can learn about one variable if we know the value of another

* https://en.wikipedia.org/wiki/Mutual_information

In [ ]:
from sklearn.metrics import mutual_info_score

In [ ]:
mutual_info_score(df_full_train.churn, df_full_train.contract)

In [ ]:
mutual_info_score(df_full_train.gender, df_full_train.churn)

In [ ]:
mutual_info_score(df_full_train.contract, df_full_train.churn)

In [ ]:
mutual_info_score(df_full_train.partner, df_full_train.churn)

In [ ]:
def mutual_info_churn_score(series):
    return mutual_info_score(series, df_full_train.churn)

In [ ]:
mi = df_full_train[categorical].apply(mutual_info_churn_score)
mi.sort_values(ascending=False)

## 3.7 Feature importance: Correlation

How about numerical columns?

* Correlation coefficient - https://en.wikipedia.org/wiki/Pearson_correlation_coefficient

In [ ]:
df_full_train.tenure.max()

In [ ]:
df_full_train[numerical].corrwith(df_full_train.churn).abs()

In [ ]:
df_full_train[df_full_train.tenure <= 2].churn.mean()

In [ ]:
df_full_train[(df_full_train.tenure > 2) & (df_full_train.tenure <= 12)].churn.mean()

In [ ]:
df_full_train[df_full_train.tenure > 12].churn.mean()

In [ ]:
df_full_train[df_full_train.monthlycharges <= 20].churn.mean()

In [ ]:
df_full_train[(df_full_train.monthlycharges > 20) & (df_full_train.monthlycharges <= 50)].churn.mean()

In [ ]:
df_full_train[df_full_train.monthlycharges > 50].churn.mean()

## 3.8 One-hot encoding

* Use Scikit-Learn to encode categorical features

In [ ]:
from sklearn.feature_extraction import DictVectorizer

In [ ]:
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

## 3.9 Logistic regression

* Binary classification
* Linear vs logistic regression

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [ ]:
z = np.linspace(-7, 7, 51)

In [ ]:
sigmoid(10000)

In [ ]:
plt.plot(z, sigmoid(z))

In [ ]:
def linear_regression(xi):
    result = w0
    
    for j in range(len(w)):
        result = result + xi[j] * w[j]
        
    return result

In [ ]:
def logistic_regression(xi):
    score = w0
    
    for j in range(len(w)):
        score = score + xi[j] * w[j]
        
    result = sigmoid(score)
    return result

## 3.10 Training logistic regression with Scikit-Learn

* Train a model with Scikit-Learn
* Apply it to the validation dataset
* Calculate the accuracy

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
model = LogisticRegression(solver='lbfgs')
# solver='lbfgs' is the default solver in newer version of sklearn
# for older versions, you need to specify it explicitly
model.fit(X_train, y_train)

In [ ]:
model.intercept_[0]

In [ ]:
model.coef_[0].round(3)

In [ ]:
y_pred = model.predict_proba(X_val)[:, 1]

In [ ]:
churn_decision = (y_pred >= 0.5)

In [ ]:
(y_val == churn_decision).mean()

In [ ]:
df_pred = pd.DataFrame()
df_pred['probability'] = y_pred
df_pred['prediction'] = churn_decision.astype(int)
df_pred['actual'] = y_val

In [ ]:
df_pred['correct'] = df_pred.prediction == df_pred.actual

In [ ]:
df_pred.correct.mean()

In [ ]:
churn_decision.astype(int)

## 3.11 Model interpretation

* Look at the coefficients
* Train a smaller model with fewer features

In [ ]:
a = [1, 2, 3, 4]
b = 'abcd'

In [ ]:
dict(zip(a, b))

In [ ]:
dict(zip(dv.get_feature_names_out(), model.coef_[0].round(3)))

In [ ]:
small = ['contract', 'tenure', 'monthlycharges']

In [ ]:
df_train[small].iloc[:10].to_dict(orient='records')

In [ ]:
dicts_train_small = df_train[small].to_dict(orient='records')
dicts_val_small = df_val[small].to_dict(orient='records')

In [ ]:
dv_small = DictVectorizer(sparse=False)
dv_small.fit(dicts_train_small)

In [ ]:
dv_small.get_feature_names_out()

In [ ]:
X_train_small = dv_small.transform(dicts_train_small)

In [ ]:
model_small = LogisticRegression(solver='lbfgs')
model_small.fit(X_train_small, y_train)

In [ ]:
w0 = model_small.intercept_[0]
w0

In [ ]:
w = model_small.coef_[0]
w.round(3)

In [ ]:
dict(zip(dv_small.get_feature_names_out(), w.round(3)))

In [ ]:
-2.47 + (-0.949) + 30 * 0.027 + 24 * (-0.036)

In [ ]:
sigmoid(_)

## 3.12 Using the model

In [ ]:
dicts_full_train = df_full_train[categorical + numerical].to_dict(orient='records')

In [ ]:
dv = DictVectorizer(sparse=False)
X_full_train = dv.fit_transform(dicts_full_train)

In [ ]:
y_full_train = df_full_train.churn.values

In [ ]:
model = LogisticRegression(solver='lbfgs')
model.fit(X_full_train, y_full_train)

In [ ]:
dicts_test = df_test[categorical + numerical].to_dict(orient='records')

In [ ]:
X_test = dv.transform(dicts_test)

In [ ]:
y_pred = model.predict_proba(X_test)[:, 1]

In [ ]:
churn_decision = (y_pred >= 0.5)

In [ ]:
(churn_decision == y_test).mean()

In [ ]:
y_test

In [ ]:
customer = dicts_test[-1]
customer

In [ ]:
X_small = dv.transform([customer])

In [ ]:
model.predict_proba(X_small)[0, 1]

In [ ]:
y_test[-1]

## 3.13 Summary

* Feature importance - risk, mutual information, correlation
* One-hot encoding can be implemented with `DictVectorizer`
* Logistic regression - linear model like linear regression
* Output of log reg - probability
* Interpretation of weights is similar to linear regression

## 3.14 Explore more

More things

* Try to exclude least useful features


Use scikit-learn in project of last week

* Re-implement train/val/test split using scikit-learn in the project from the last week
* Also, instead of our own linear regression, use `LinearRegression` (not regularized) and `RidgeRegression` (regularized). Find the best regularization parameter for Ridge

Other projects

* Lead scoring - https://www.kaggle.com/ashydv/leads-dataset
* Default prediction - https://archive.ics.uci.edu/ml/datasets/default+of+credit+card+clients

